In [1]:
import csv
import numpy as np
import pandas as pd
import sys, os
import re
import ast
import datetime as dt
from tqdm import tqdm
from collections import defaultdict, Counter

import warnings

# 禁用所有警告
warnings.filterwarnings("ignore")

In [2]:
def median_proc_process(ts):
    # 将所有列值全部为 0 的列替换为 NaN
    ts.loc[:, (ts == 0).all(axis=0)] = np.nan  
    
    # 处理列数据
    processed_values = {}
    for col in ts.columns:
        if col.startswith('PROC_Cate_'):  
            # 对于 'CHART_Cate_' 开头的列，取最大值
            processed_values[col] = ts[col].max()
        elif col.startswith('PROC_Str_'):  
            # 对于 'CHART_Str_' 开头的列，取除 0 外出现次数最多的值
            non_zero_values = ts[col][ts[col] != '0']
            if not non_zero_values.empty:
                processed_values[col] = non_zero_values.mode().iloc[0]
            else:
                processed_values[col] = 0
        else:
            # 其他列取中位数
            processed_values[col] = ts[col].median()
    
    # 将结果转为 DataFrame
    result = pd.DataFrame([processed_values])
    
    return result

In [4]:
# 24
all_ts = pd.DataFrame()

# 遍历每个 ICUSTAY_ID
for i in tqdm(os.listdir('D:/2024BI_Journal/MIMIC_IV/Final_Proc_2/')):
    try:
        # 读取 dynamic.csv 文件
        file_path = f'D:/2024BI_Journal/MIMIC_IV/Final_Proc_2/{i}'
        ts = pd.read_csv(file_path)
        
        if ts.shape[0]>0:
            ts = ts.iloc[:96,:]

            # 调用 ts_process 函数进行数据处理
            ts = median_proc_process(ts)

            ts['ICUSTAY_ID'] = i.split('.')[0]

            # 合并处理后的数据
            all_ts = pd.concat([all_ts, ts], ignore_index=True)
    except Exception as e:
        # 打印错误信息并跳过该文件
        print(f"Error processing file {i}: {e}")
    
all_ts = all_ts.dropna(axis=1, how='all')

100%|████████████████████████████████████████████████████████████████████████████| 57475/57475 [27:38<00:00, 34.65it/s]


In [5]:
all_ts.shape

(57475, 54)

In [8]:
all_ts.head(2)

,PROC_Cate_225402,PROC_225752,PROC_Cate_221214,PROC_Cate_225433,PROC_Cate_227194,ICUSTAY_ID,PROC_225792,PROC_229351,PROC_Cate_225479,PROC_Cate_225401,...,PROC_Cate_225437,PROC_Cate_225475,PROC_224273,PROC_Cate_227713,PROC_Cate_225467,PROC_Cate_225449,PROC_Cate_225472,PROC_Cate_225450,PROC_Cate_225442,PROC_Cate_226237
0,1.0,1785.0,1.0,1.0,1.0,30000153,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,30000213,1614.0,1697.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
all_ts.to_csv('D:/2024BI_Journal/MIMIC_IV/Median_B/A24_Proc_median_max_nan.csv',index=False)